# Tree Death Detection

In [2]:
#Imports the drive module from Google Colab’s library, which has tools for interacting with Google Drive.
from google.colab import drive
#Mounts (connects) your Google Drive to the Colab notebook at the folder path /content/drive.

drive.mount('/content/drive', force_remount=True)


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Creates to directory where all of the files realintg to the model will be stored
dir_root = '/content/drive/MyDrive/Tree_Mortality' # make sure the data folder is correct


In [ ]:
!pip install rasterio

In [ ]:
import os
import numpy as np
import time
import imageio.v2 as imageio
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
import rasterio
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import tifffile as tiff

In [ ]:
#Defines a function named read_tif that takes one argument fname (the file path to a .tif raster file).
def read_tif(fname):
  #Opens the raster file at path fname in read mode ("r") using rasterio.
  with rasterio.open(fname, "r") as src:
    #Reads all the bands of the raster into a NumPy array.
    data = src.read()
    #If the raster has multiple bands, data will be a 3D array with shape (bands, height, width).
#Returns the NumPy array containing the raster data so you can use it later in your code.
  return data





In [ ]:
# preview data
tmp_img = read_tif(dir_root+'/Training/studyarea77.tif')
print(tmp_img.shape, tmp_img.dtype)

In [ ]:

# Plots data to make sure the labels are good
plt.figure(figsize=(10,15))

plt.subplot(131)
plt.imshow(tmp_img[:3].transpose([1,2,0]) / 255)

plt.subplot(132)
plt.imshow(tmp_img[-2])

plt.subplot(133)
plt.imshow(tmp_img[-1])

plt.show()

# Split Train/Val/Test

In [ ]:
import tensorflow as tf

In [ ]:
# Sets the size of image patches and number of channels:
# IMG_HEIGHT × IMG_WIDTH = size of each patch.
# IMG_CHANNEL = 5 → first 4 channels are inputs, last one may be the target (label).
# OUT_CHANNEL = 1 → number of channels in the label.
IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNEL = 5
OUT_CHANNEL = 1

# Converts a NumPy array into a TFRecord float feature.
# Flattens the array (reshape(-1)) because TFRecords store data as 1D lists.
def _float_feature(value):
  return tf.train.Feature(
    float_list=tf.train.FloatList(value=value.reshape(-1))
  )
# Converts a NumPy array into a TFRecord float feature.
# Flattens the array (reshape(-1)) because TFRecords store data as 1D lists.
def feature_example(image, label):
  feature = {
    'label': _float_feature(label),
    'image': _float_feature(image),
  }

  return tf.train.Example(features=tf.train.Features(feature=feature))

#   ls_filepath → list of raster file paths.
# output_name → output TFRecord file path.
# step=128 → stride for extracting overlapping patches.
def generate_data(ls_filepath, output_name, step=128):

#Opens a TFRecord writer to save processed data.
  with tf.io.TFRecordWriter(output_name) as all_writer:
  # Reads the raster using read_tif.
# Converts to float32.
# Normalizes the first IMG_CHANNEL-1 channels to [0,1] by dividing by 255.
# Normalizes the last channel (probably NDVI or another index) from [-1,1] to [0,1].
# Transposes the array from (bands, height, width) → (height, width, bands) for TensorFlow.
    for file in tqdm(ls_filepath):
      data_inp = read_tif(file)
      data_inp = data_inp.astype(np.float32)
      data_inp[:IMG_CHANNEL-1] = data_inp[:IMG_CHANNEL-1] / 255
      data_inp[IMG_CHANNEL-1] = (data_inp[IMG_CHANNEL-1] + 1) / 2
      data_inp = np.transpose(data_inp, (1, 2, 0))

      nan_mask = np.any(np.isnan(data_inp), axis=-1)
      data_inp[nan_mask, :] = 0
# Sets all pixels with NaNs to 0.
      img_xsize, img_ysize, _ = data_inp.shape
      for x in range(0, img_xsize, step):
        for y in range(0, img_ysize, step):
          x_end = min(x + IMG_HEIGHT, img_xsize)
          y_end = min(y + IMG_WIDTH, img_ysize)
          patch = data_inp[x:x_end, y:y_end, :IMG_CHANNEL]
          label_patch = data_inp[x:x_end, y:y_end, -OUT_CHANNEL:]
# Slides a window of size IMG_HEIGHT × IMG_WIDTH across the raster.
# Stride is step (can produce overlapping patches).
# Extracts input channels and label channels separately.

          # Apply padding if patch is smaller than IMG_HEIGHT or IMG_WIDTH
          if patch.shape[0] < IMG_HEIGHT or patch.shape[1] < IMG_WIDTH:
            patch = np.pad(
              patch,
              ((0, IMG_HEIGHT - patch.shape[0]), (0, IMG_WIDTH - patch.shape[1]), (0, 0)),
              mode='constant', constant_values=0
            )

            label_patch = np.pad(
              label_patch,
              ((0, IMG_HEIGHT - label_patch.shape[0]), (0, IMG_WIDTH - label_patch.shape[1]), (0, 0)),
              mode='constant', constant_values=0
            )

             #  For patches at the edges that are smaller than the desired size, pads them with zeros.

          # # comment out when padding is added on 1/16/2025
          # if (x+IMG_HEIGHT > img_xsize) or (y+IMG_WIDTH > img_ysize):
          #   continue

          tf_example = feature_example(patch, label_patch)
          all_writer.write(tf_example.SerializeToString())
# Converts the patch and label to a tf.train.Example.
# Serializes and writes it to the TFRecord file.
      del data_inp
      # Deletes the large array from memory to save RAM.


In [ ]:
train_files = []



test_files = [
]



train_tfrname = dir_root+'/train.tfrecord'
test_tfrname = dir_root+'/test.tfrecord' #train_tfrname and test_tfrname are just the filenames you’ll save the TensorFlow records to.

In [ ]:
if os.path.exists(train_tfrname):
  print(f'training data already cropped to {train_tfrname}') #Checks if the training TFRecord file exists
else:
  generate_data(train_files, train_tfrname, step=128)
  generate_data(test_files, test_tfrname, step=128)
  #Otherwise, it generates the TFRecord datasets:
  #If the file doesn’t exist, it calls generate_data() to crop the training and testing .tif images into smaller patches (step size = 128) and saves them as .tfrecord files.
generate_data(train_files, train_tfrname, step=128)
generate_data(test_files, test_tfrname, step=128)
# Then, regardless of the check, it runs again:
# This means both train and test data are generated every time you run the cell, whether or not they already exist.


# Train

In [ ]:
def input_pipeline(filename, batch_size, is_train=True, is_shuffle=True):
    feature_description = {
        'label': tf.io.VarLenFeature(tf.float32),
        'image': tf.io.VarLenFeature(tf.float32),
    }
# The TFRecords contain two fields:
# "image" → the input raster image patch (float values).
# "label" → the target label patch (float values).
# They’re stored as sparse tensors, so later you convert them to dense arrays.
    # Data augmentation (training only)
    def _augTrain(img_and_label):
        img_and_label = tf.image.rot90(img_and_label, tf.random.uniform([], 0, 4, dtype=tf.int32))
        img_and_label = tf.image.random_flip_left_right(img_and_label)
        img_and_label = tf.image.random_flip_up_down(img_and_label)
        return img_and_label
#Each image/label pair can be randomly rotated, flipped left/right, or flipped up/down.
# This helps prevent overfitting.
    def _process_train(inp_img, inp_label):
        img_and_label = tf.concat([inp_img, inp_label], axis=-1)
        img_and_label = _augTrain(img_and_label)
        inp_img = img_and_label[..., :inp_img.shape[-1]]
        inp_label = img_and_label[..., inp_img.shape[-1]:]
        return inp_img, inp_label
# This makes sure the label rotates/flips in the exact same way as the image (so they still match).
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)

        image = tf.sparse.to_dense(feature_dict['image'], default_value=0)
        image = tf.reshape(image, [IMG_HEIGHT, IMG_WIDTH, IMG_CHANNEL])  # <-- replace with your numbers

        label = tf.sparse.to_dense(feature_dict['label'], default_value=0)
        label = tf.reshape(label, [IMG_HEIGHT, IMG_WIDTH, OUT_CHANNEL])  # <-- replace with your numbers

        if is_train:
            image, label = _process_train(image, label)

        label = label[..., 0]  # squeeze last channel
        return image, label
# Reads TFRecord → converts to dense → reshapes into image/label arrays → applies augmentation if training → final output = (image, label) pair.
    # Build dataset pipeline
    dataset = tf.data.TFRecordDataset(filename)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=200)
    batch = (
        dataset
        .map(_parse_function)
        .batch(batch_size)
        .prefetch(buffer_size=tf.data.AUTOTUNE)
    )

    return batch
# Reads all TFRecords, shuffles them (if requested), parses them, batches them, and prefetches for performance.

In [ ]:
BATCH_SIZE = 12
# the pipeline will group the data into sets of 2 samples at a time.

In [ ]:
tmp_tfds = input_pipeline(train_tfrname, BATCH_SIZE, is_train=True, is_shuffle=False)
# Creates a dataset pipeline from your training TFRecord:
# BATCH_SIZE = 2 → each batch contains 2 image patches and 2 label patches.
# is_train=True → augmentation (rotation/flip) will be applied.
# is_shuffle=False → data is read in order, no shuffling.
for x, y in tmp_tfds:
  print(x.shape, y.shape)
  break

  # Iterates through the dataset and prints the shape of the first batch:
  # x → batch of images
  # y → batch of labels
  # break → stops after the first batch, so you only see one output.


  # Expected output shapes:
  # x.shape → (2, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNEL)
  # y.shape → (2, IMG_HEIGHT, IMG_WIDTH)

In [ ]:
i = 0

plt.figure(figsize=(10,15))

plt.subplot(131)
plt.imshow(x[i,...,:3])

plt.subplot(132)
plt.imshow(x[i,...,-1])

plt.subplot(133)
plt.imshow(y[i])

plt.show()

# Selects the 1st sample in the batch for plotting.

In [ ]:
def down_block(x, filters, use_maxpool=True, activation='swish', dropout_rate=0.3):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    if use_maxpool:
        return tf.keras.layers.MaxPooling2D(strides=(2,2))(x), x
    else:
        return x

def up_block(x, y, filters, activation='swish', dropout_rate=0.3):
    x = tf.keras.layers.UpSampling2D()(x)
    x = tf.keras.layers.Concatenate(axis=3)([x, y])
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    return x

def Unet_v2(input_size=(256,256,5), num_class=2, dropout_rate=0.4, activation='swish'):
    filters = [64, 128, 256, 512, 1024]
    input = tf.keras.layers.Input(shape=input_size)

    x, t1 = down_block(input, filters[0], activation=activation)
    x, t2 = down_block(x, filters[1], activation=activation)
    x, t3 = down_block(x, filters[2], activation=activation)
    x, t4 = down_block(x, filters[3], activation=activation)
    x = down_block(x, filters[4], use_maxpool=False, activation=activation)

    x = up_block(x, t4, filters[3], activation=activation)
    x = up_block(x, t3, filters[2], activation=activation)
    x = up_block(x, t2, filters[1], activation=activation)
    x = up_block(x, t1, filters[0], activation=activation)

    x = tf.keras.layers.Dropout(dropout_rate)(x)
    if num_class > 0:
        output = tf.keras.layers.Conv2D(num_class, 1, activation=None)(x)
    else:
        output = x
    return tf.keras.Model(input, output, name='unet_v2')

# This function builds a U-Net model:


In [ ]:


# Theese are the functions to calculate the loss
@tf.function
def dice_loss(y_true, y_pred, ep=1e-8):
    intersection = tf.reduce_sum(y_pred[...,1] * y_true) + ep
    union = tf.reduce_sum(y_pred[...,1]) + 2 * tf.reduce_sum(y_true) + ep
    return 1 - 2 * intersection / union

@tf.function
def focal_loss(y_true, y_pred, gamma=2., alpha=0.25):
    pt_1 = tf.where(tf.equal(y_true, 1), y_pred[...,1], tf.ones_like(y_pred[...,1]))
    pt_0 = tf.where(tf.equal(y_true, 0), y_pred[...,1], tf.zeros_like(y_pred[...,1]))
    loss = -tf.reduce_sum(alpha * tf.pow(1.-pt_1, gamma) * tf.math.log(pt_1 + 1e-8)) \
           -tf.reduce_sum((1-alpha) * tf.pow(pt_0, gamma) * tf.math.log(1.-pt_0 + 1e-8))
    return loss

def hybrid_loss(y_true, y_pred, alpha=0.5):
    return alpha * dice_loss(y_true, y_pred) + (1-alpha) * focal_loss(y_true, y_pred)


In [ ]:
@tf.function
def train_step(tf_model, tf_opmz, x, y):
  with tf.GradientTape() as u_tape:
    y_ = tf_model(x, training=True)
# Forward pass:U-Net outputs logits (unnormalized scores).

    # v5 version with dice loss
    y_ = tf.nn.softmax(y_) # [B, H, W, 2]
    # Convert to probabilities:
    # Assumes 2-class segmentation (background + foreground).
    loss = dice_loss(tf.cast(y, tf.float32), y_)
    # Loss calculation: y should be binary masks (shape [B,H,W] or [B,H,W,1] with values 0/1).
    # If [B,H,W], broadcasting will work fine.
    # If [B,H,W,1], might need tf.squeeze(y, axis=-1).
  grad_u_model = u_tape.gradient(loss, tf_model.trainable_variables)
  tf_opmz.apply_gradients(zip(grad_u_model, tf_model.trainable_variables))
  # Backpropagation & update
  return y_, [loss]
  # Runs one step of training: forward pass → loss → backpropagation → weight update.
  # Returns the predicted mask (y_) and the loss.
@tf.function
def val_step(tf_model, x, y):
  y_ = tf_model(x, training=False)

  # v5 version with dice loss
  y_ = tf.nn.softmax(y_) # [B, H, W, 2]
  loss = dice_loss(tf.cast(y, tf.float32), y_)


  return y_, [loss]
  # Same as training step but does not update weights.
  # Used to check model performance on validation data.

def computeKerasF1Score(tp, fp, fn):
  np_pre = tp.result() / (tp.result() + fp.result())
  np_rec = tp.result() / (tp.result() + fn.result())
  np_f1 = 2 * (np_pre * np_rec) / (np_pre + np_rec)

  return np_f1.numpy(), np_pre.numpy(), np_rec.numpy()

#   Takes true positives, false positives, false negatives (from metrics you collect).
# Computes precision, recall, and F1 score.
# Converts them to NumPy numbers so you can log or print them.

In [ ]:
def train_val_step(tf_model, tf_opmz, tf_ckp, output_ckp, train_path, val_path, epoch, save_step):
    t_s = time.time()
    res_acc = [epoch+1]

    train_batch = input_pipeline(train_path, BATCH_SIZE, is_train=True, is_shuffle=True)
    val_batch = input_pipeline(val_path, BATCH_SIZE, is_train=False, is_shuffle=True)

    # train
    train_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    train_loss = []
    for train_img, train_lab in train_batch:
        train_pred, train_loss_sub = train_step(tf_model, tf_opmz, train_img, train_lab)

        train_acc.update_state(train_lab, train_pred)

        train_loss.append(train_loss_sub)
    train_loss = tf.reshape(tf.concat(train_loss, axis=0), (-1, len(train_loss_sub)))
    train_loss = tf.reduce_mean(train_loss, axis=0).numpy()

    np_train_acc = train_acc.result().numpy()
    res_acc += [train_loss, np_train_acc, 0, 0, 0]

    train_acc.reset_state()


    # test
    test_acc = tf.keras.metrics.SparseCategoricalAccuracy()
    test_loss = []
    for test_img, test_lab in val_batch:
        test_pred, test_loss_sub = val_step(tf_model, test_img, test_lab)

        test_acc.update_state(test_lab, test_pred)

        test_loss.append(test_loss_sub)
    test_loss = tf.reshape(tf.concat(test_loss, axis=0), (-1, len(test_loss_sub)))
    test_loss = tf.reduce_mean(test_loss, axis=0).numpy()

    np_test_acc = test_acc.result().numpy()
    res_acc += [test_loss, np_test_acc, 0, 0, 0]

    test_acc.reset_state()


    # output
    if (epoch + 1) % save_step == 0:
        tf_ckp.save(file_prefix = os.path.join(output_ckp, 'ckp'+str(epoch+1)))

    print(f'Epoch {res_acc[0]} '+
        f'spent {(time.time()-t_s)/60:.2f} mins: '+

        f'train loss = {train_loss[0]:.4f}, '+
        f'acc = {np_train_acc:.4f};  '+

        f'val loss = {test_loss[0]:.4f}, '+
        f'acc = {np_test_acc:.4f}.')

    return res_acc
# This function runs one epoch of training and validation.
# Inputs:
  # tf_model: your neural network.
  # tf_opmz: optimizer (e.g., Adam).
  # tf_ckp: TensorFlow checkpoint manager.
  # output_ckp: directory to save checkpoints.
  # train_path, val_path: TFRecord paths for training/validation.
  # epoch: current epoch index.
  # save_step: how often to save checkpoints.
# Process:
    # Creates training and validation datasets (train_batch, val_batch).
    # Initializes metrics (SparseCategoricalAccuracy).
# Training loop:
  # For each batch, calls train_step (computes forward pass, loss, and applies gradients).
  # Tracks training accuracy + loss.
# Validation loop:
  # For each batch, calls val_step (forward pass only).
  # Tracks validation accuracy + loss.
  # Saves checkpoint every save_step epochs.
# Prints summary for that epoch

def plotHist(res_acc, str_train='train_acc', str_val='val_acc', isMax=True, output_path=''):
    plt.figure()

    plt.plot(res_acc[str_train], alpha=0.5, label=str_train)
    plt.plot(res_acc[str_val], alpha=0.5, label=str_val)

    if isMax:
        y_val = res_acc[str_val].max()
        x_idx = res_acc[str_val].idxmax()
    else:
        y_val = res_acc[str_val].min()
        x_idx = res_acc[str_val].idxmin()

    plt.xlabel('epoch')
    plt.legend()
    plt.tight_layout()
    if len(output_path) > 0:
        plt.savefig(output_path)
# Plots training vs validation metrics across epochs.
# Can plot accuracy curves or loss curves.
# Marks the best point (min loss or max accuracy).
# Optionally saves the figure.

def train(tf_model, tf_opmz, ls_ckp, output_ckp, train_path, val_path, epochs, initial_epoch=0, save_step=5):
    acc_column = ['epoch','train_loss','train_acc','train_f1','train_pre','train_rec',
                  'val_loss','val_acc','val_f1','val_pre','val_rec']
    if initial_epoch != 0:
        res_acc = pd.read_csv(os.path.join(os.path.dirname(output_ckp), 'loss.csv'))

    min_f1 = 0
    for i in range(initial_epoch, initial_epoch + epochs):

        cur_acc = train_val_step(tf_model, tf_opmz, ls_ckp[0], output_ckp, train_path, val_path, i, save_step)


        cur_acc = pd.DataFrame([cur_acc], columns=acc_column)
        if i != 0:
            res_acc = pd.concat([res_acc, cur_acc], ignore_index=True)
        else:
            res_acc = cur_acc
        res_acc.to_csv(os.path.join(os.path.dirname(output_ckp), 'loss.csv'), index=False)

    plotHist(res_acc, str_train='train_loss', str_val='val_loss', isMax=False,
             output_path=os.path.join(os.path.dirname(output_ckp),'loss.png'))


In [ ]:
MODEL_NAME = 'UNet_v0'

dir_result = os.path.join(dir_root, 'Results', MODEL_NAME)
# Creates a folder path
# This is the main results folder for this specific model.
dir_ckp = os.path.join(dir_result, 'ckp')
# Creates another folder inside it
# This is where checkpoints (saved model weights during training) will be stored.
if not os.path.exists(dir_ckp):
    os.makedirs(dir_ckp)
# Makes sure the checkpoint folder exists.
# If it doesn’t, it creates it automatically.

In [ ]:
# del u_model

u_model = Unet_v2(input_size=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNEL),
                  num_class=2, dropout_rate=0.4, activation='swish')


# Creates a new U-Net model with:


initial_learning_rate = 1e-4
decay_steps = 1000 # Number of steps before decay
decay_rate = 0.98 # Decay factor per step
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate, decay_steps, decay_rate, staircase=True )

# Starts training with a very small learning rate (5×10⁻⁶).
# Every 1000 training steps → multiply LR by 0.98.
# staircase=True → decay happens in discrete steps, not continuously.

u_optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
# Uses Adam optimizer, but instead of a fixed learning rate, it follows the lr_schedule above.


u_model.compile(optimizer=u_optimizer, loss=hybrid_loss, metrics=['accuracy'])


ckp = tf.train.Checkpoint(u_optimizer=u_optimizer, u_model=u_model)
ckp_mng = tf.train.CheckpointManager(ckp, directory=dir_ckp, max_to_keep=10, checkpoint_name='best')

# ckp bundles together the model + optimizer state so both can be restored later.
# CheckpointManager:
# Saves checkpoints in the dir_ckp folder.
# Keeps only the latest 10 checkpoints (older ones auto-deleted).


In [ ]:
dataset = tf.data.TFRecordDataset(train_tfrname)

count = sum(for _ in dataset)``
print(f"Total number of samples: {count}")

# This code loads your training data file (train_tfrname), goes through every entry inside it, counts how many there are, and then prints the total number of samples in that dataset.

In [ ]:
latest_checkpoint = ckp_mng.latest_checkpoint


if latest_checkpoint:
    print(f"Restoring from {latest_checkpoint}")
    ckp.restore(latest_checkpoint)
    initial_epoch = int(latest_checkpoint.split('ckp')[-1].split('-')[0])
    print(f"Resuming training from epoch {initial_epoch}")

    train(u_model, u_optimizer, [ckp, ckp_mng], dir_ckp,
          train_tfrname, test_tfrname,
          epochs=30, initial_epoch=initial_epoch, save_step=1)
else:
  print("No checkpoint found. Starting training from scratch.")
  train(u_model, u_optimizer, [ckp, ckp_mng], dir_ckp,
          train_tfrname, test_tfrname,
          epochs=50, initial_epoch=0, save_step=1)


# Evaluation

In [ ]:
ckp_chosen = os.path.join(dir_ckp, 'ckp31-31')

#builds a full file path by combining your checkpoint folder (dir_ckp) with the string 'ckp23-23'.


print(ckp_chosen)

In [ ]:
u_model_unfixed = Unet_v2(input_size = (None,None,IMG_CHANNEL), num_class=2, dropout_rate=False)
# creates a new U-Net model, but with input_size=(None,None,IMG_CHANNEL) so it can accept images of any spatial size (height/width don’t need to be fixed).

ckp_unfixed = tf.train.Checkpoint(u_model=u_model_unfixed)
# sets up a TensorFlow checkpoint manager tied to this model.

print('restore ckp from :', ckp_chosen)
ckp_unfixed.restore(ckp_chosen)
# loads the saved weights from that checkpoint into your new model.


In [ ]:
def prepare_data(data_fname):
  data_inp = read_tif(data_fname)
  data_inp = data_inp.astype(np.float32)
  data_inp[:IMG_CHANNEL-1] = data_inp[:IMG_CHANNEL-1] / 255
  data_inp[IMG_CHANNEL-1] = (data_inp[IMG_CHANNEL-1] + 1) / 2
  data_inp = np.transpose(data_inp, (1, 2, 0))

  nan_mask = np.any(np.isnan(data_inp), axis=-1)
  data_inp[nan_mask, :] = 0

  data_inp = np.expand_dims(data_inp, axis=0)

  return data_inp[...,:IMG_CHANNEL], data_inp[...,-OUT_CHANNEL:], nan_mask

# Reads a .tif image using read_tif(data_fname).
# Converts pixel values to float32 and normalizes:
# First IMG_CHANNEL-1 channels → divide by 255 (standard image normalization).
# Last channel → scaled to range [0,1] via (x + 1)/2.
# Transposes the array so channel dimension is last (H×W×C).
# Replaces any NaN values with 0.
# Adds a batch dimension (1×H×W×C).


# Returns:
# Input channels: [..., :IMG_CHANNEL]
# Label channels: [..., -OUT_CHANNEL:]


def pad_to_multiple_of_n(img, n=16):
  pad_x = (-img.shape[1]) % n
  pad_y = (-img.shape[2]) % n

  pad_xu = pad_x // 2
  pad_xd = pad_x - pad_xu
  pad_yl = pad_y // 2
  pad_yr = pad_y - pad_yl

  padded_img = np.pad(img, pad_width=
    ((0,0), (pad_xu, pad_xd),
    (pad_yl, pad_yr), (0, 0)),
    mode='constant', constant_values=0)
    # mode='reflect')
    # mode='edge')

  return padded_img, (pad_xu, pad_yl), (pad_xd, pad_yr)
# Pads an image so its height and width are multiples of n (default 16).
# Padding is applied symmetrically on both sides.
# Returns:
  # The padded image.
  # Tuple (pad_xu, pad_yl) → padding added to top/left.
  # Tuple (pad_xd, pad_yr) → padding added to bottom/right.

def read_file_to_list(file_path):
  with open(file_path, 'r') as f:
    return f.read().splitlines()
# Reads a text file line by line.
# Returns a list of strings, one per line.


In [ ]:
# output prediction result here
dir_output = os.path.join(dir_root, MODEL_NAME, 'pred_ckp23')
# Creates a path
os.makedirs(dir_output, exist_ok=True)
# Creates the folder if it doesn’t already exist.
print(f'Ouput to {dir_output}')
print(f'Total image : {len(test_files)}')

In [ ]:
from skimage.transform import resize

scale_factor = 0.5

for cur_file in tqdm(test_files):

# Iterates through all test .tif files, showing a progress bar with tqdm.

  data_inp, data_label, nan_mask = prepare_data(cur_file)
# normalizes the image and separates input vs. label.
   # --- Resize step ---
  new_shape = (
        int(data_inp.shape[1] * scale_factor),  # height
        int(data_inp.shape[2] * scale_factor)   # width
    )

  data_inp = resize(
        data_inp[0], new_shape, order=1, preserve_range=True, anti_aliasing=True
    )[np.newaxis, ...]  # back to shape (1,H,W,C)

  data_label = resize(
        data_label[0], new_shape, order=0, preserve_range=True, anti_aliasing=False
    )[np.newaxis, ...]  # keep labels discrete (order=0)

  nan_mask = resize(
        nan_mask, new_shape, order=0, preserve_range=True, anti_aliasing=False
    ).astype(bool)

  data_inp, (pad_xu, pad_yl), (pad_xd, pad_yr) = pad_to_multiple_of_n(data_inp, 16)
# pads the image so its height/width are multiples of 16.
  data_inp = data_inp.astype(np.float32)
# Ensures the input type is compatible with TensorFlow.



  pred_out = u_model_unfixed(data_inp, training=False)
  pred_out = tf.argmax(pred_out, axis=-1).numpy().astype(np.float32)
# Passes the input through the restored model.
# Converts the output probabilities to class labels (0 or 1) using argmax.

  pred = pred_out[0, pad_xu:pred_out.shape[1]-pad_xd, pad_yl:pred_out.shape[2]-pad_yr]
  pred[nan_mask] = np.nan
# Removes padding to match original image size.
# Sets pixels that were originally NaN back to NaN.

  res = np.dstack([data_label.squeeze(), pred.squeeze()])
# Stacks the true label and predicted label for saving/plotting.
  np.save(os.path.join(dir_output, Path(cur_file).stem+'_pred.npy'), res)
# Saves the res array as a .npy file in the output folder.

  # plot
  fig, axs = plt.subplots(1, 3, figsize=(15, 5))

  axs[0].imshow(data_inp[0,...,:3])
  axs[0].set_title('RGB')

  axs[1].imshow(res[...,0])
  axs[1].set_title('True')

  axs[2].imshow(res[...,1])
  axs[2].set_title('Pred')

  fig.suptitle(Path(cur_file).stem, fontsize=16)
  plt.tight_layout()
  plt.savefig(os.path.join(dir_output, Path(cur_file).stem+'_pred'), dpi=300)


In [ ]:
# load prediction data
ls_output = glob(os.path.join(dir_output, '*.npy'))
print(f'Number of test files = {len(ls_output)}')

np_output = [np.load(i).reshape(-1,2) for i in ls_output]
np_output = np.concatenate(np_output, axis=0)
print(np_output.shape)

In [ ]:
# Check for NaNs
print("NaNs in labels:", np.isnan(np_output[:,0]).sum())
print("NaNs in preds:", np.isnan(np_output[:,1]).sum())

# Remove rows containing NaNs
mask = ~np.isnan(np_output).any(axis=1)
np_output_clean = np_output[mask]



# # compute f1, precision, recall, accuracy
# f1_glob = f1_score(np_output[:,0], np_output[:,1], average=None)
# pre_glob = precision_score(np_output[:,0], np_output[:,1], average=None)
# rec_glob = recall_score(np_output[:,0], np_output[:,1], average=None)
# acc_glob = accuracy_score(np_output[:,0], np_output[:,1])

# Compute F1, precision, recall, accuracy (weighted to account for class imbalance)
f1_glob = f1_score(np_output_clean[:,0], np_output_clean[:,1], average='weighted')
pre_glob = precision_score(np_output_clean[:,0], np_output_clean[:,1], average='weighted')
rec_glob = recall_score(np_output_clean[:,0], np_output_clean[:,1], average='weighted')
acc_glob = accuracy_score(np_output_clean[:,0], np_output_clean[:,1])

# Print results
print('====Global evaluation ====')
print('f1 = ', f1_glob)
print('pre = ', pre_glob)
print('rec = ', rec_glob)
print('accuracy = ', acc_glob)

# Save results to file
filename = os.path.join(dir_result, 'global_eval_' +
                        os.path.basename(ckp_chosen).split('-')[0] + '.txt')
with open(filename, 'w') as file:
    file.write('====Global evaluation for changes ====\n')
    file.write(f'f1 = {f1_glob}\n')
    file.write(f'pre = {pre_glob}\n')
    file.write(f'rec = {rec_glob}\n')
    file.write(f'accuracy = {acc_glob}\n')

print(f'Results have been written to {filename}')
